# CatBoost: pre-bid risk

Run `python preprocessing.py` once from the project folder. The input has one row per auction, and IsBadBuy = 1 is the positive class. Compare the two objectives on the same validation cars using continuous scores.

In [ ]:
import pandas as pd
from preprocessing import ROOT, load_development
from evaluation import save_validation

# Selected 15-field configuration; use False only to reproduce the 11-field baseline.
include_prices = True
X_train, y_train, X_val, y_val = load_development(include_prices=include_prices)
result_dir = ROOT / "results" / "validation"
if include_prices:
    result_dir = result_dir / "acquisition_prices"
scores = pd.DataFrame(index=y_val.index)
print(f"{len(X_train):,} training cars, {len(X_val):,} validation cars")

### Matched boosted-tree settings
Both models receive the same named categorical fields. CatBoost encodes them during fitting. Tree and sampling settings are fixed in advance.

In [ ]:
from tree_models import BASELINE, make_tree

depth, l2_regularization = BASELINE["catboost"]

### Fit squared-error and log-loss versions
RMSE fits the binary outcome as a real-valued target. Keep its raw score for AUC; it need not be a probability.

In [ ]:
cb_l2 = make_tree("CatBoost_L2", depth, l2_regularization)
cb_l2.fit(X_train, y_train)
scores["CatBoost_L2"] = cb_l2.predict(X_val)

cb_log = make_tree("CatBoost_Logloss", depth, l2_regularization)
cb_log.fit(X_train, y_train)
scores["CatBoost_Logloss"] = cb_log.predict_proba(X_val)[:, 1]

### Validation AUC
Use the shared evaluator and save continuous scores indexed by RefId.

In [ ]:
save_validation(y_val, scores, result_dir, "catboost")

### Hyperparameter tuning

Run `%run tune_trees.py catboost` in this notebook to screen the six settings for each objective and refit the finalists. After both families finish, run `%run tune_trees.py summarize` for the validation AUC comparison and bidding thresholds.